In [ ]:
# =========================================================
# OPTIONAL: P100 COMPATIBLE PYTORCH SETUP
# =========================================================

# Google Colab usually provides a PyTorch build that works with its GPUs.
# Keep this disabled for Colab. Enable it only if a Kaggle Tesla P100 session
# explicitly fails with: sm_60 is not compatible with the current PyTorch build.

ENABLE_P100_TORCH_FIX = False
FORCE_REINSTALL_P100_TORCH = False
VERIFY_TORCH_INSTALL_IN_SUBPROCESS = True

import os
import sys
import time
import subprocess
from pathlib import Path

IS_COLAB_BOOTSTRAP = "google.colab" in sys.modules or Path("/content").exists()
IS_KAGGLE_BOOTSTRAP = Path("/kaggle/input").exists()
P100_SETUP_MARKER = Path("/kaggle/working/.p100_torch_cu126_installed") if IS_KAGGLE_BOOTSTRAP else Path("/content/.p100_torch_cu126_installed")


def detect_gpu_names_with_nvidia_smi():
    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=name",
                "--format=csv,noheader"
            ],
            check=False,
            capture_output=True,
            text=True
        )
    except FileNotFoundError:
        return []

    if result.returncode != 0:
        return []

    return [line.strip() for line in result.stdout.splitlines() if line.strip()]


def restart_kernel_now():
    print("[INFO] Restarting the kernel automatically...")

    try:
        from IPython import get_ipython

        ipython = get_ipython()
        kernel = getattr(ipython, "kernel", None)

        if kernel is not None:
            kernel.do_shutdown(restart=True)
            time.sleep(3)
    except Exception as error:
        print(f"[WARN] Soft restart request failed: {error}")

    os._exit(0)


def verify_torch_install_in_subprocess():
    if not VERIFY_TORCH_INSTALL_IN_SUBPROCESS:
        return

    verify_code = """
import torch
print('[VERIFY] torch version:', torch.__version__)
print('[VERIFY] torch CUDA build:', torch.version.cuda)
print('[VERIFY] CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('[VERIFY] GPU:', torch.cuda.get_device_name(0))
    print('[VERIFY] capability:', torch.cuda.get_device_capability(0))
    print('[VERIFY] supported arch list:', torch.cuda.get_arch_list())
"""

    result = subprocess.run(
        [sys.executable, "-c", verify_code],
        check=False,
        capture_output=True,
        text=True
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("The new torch installation could not be verified.")


gpu_names = detect_gpu_names_with_nvidia_smi()
p100_detected = any("P100" in gpu_name for gpu_name in gpu_names)

print(f"[INFO] Colab environment: {IS_COLAB_BOOTSTRAP}")
print(f"[INFO] Kaggle environment: {IS_KAGGLE_BOOTSTRAP}")
print(f"[INFO] GPUs detected before torch import: {gpu_names if gpu_names else 'none'}")
print(f"[INFO] P100 detected: {p100_detected}")

if not ENABLE_P100_TORCH_FIX:
    print("[INFO] P100 torch replacement skipped. This is the recommended setting for Google Colab.")
elif p100_detected and "torch" in sys.modules:
    print("[WARN] torch is already imported in this kernel, so PyTorch cannot be replaced safely.")
    restart_kernel_now()
elif p100_detected and (FORCE_REINSTALL_P100_TORCH or not P100_SETUP_MARKER.exists()):
    print("[INFO] Installing PyTorch CUDA 12.6 build for Tesla P100/sm_60...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torch",
        "torchvision",
        "torchaudio"
    ])

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "torch==2.11.0",
        "torchvision==0.26.0",
        "torchaudio==2.11.0",
        "--index-url",
        "https://download.pytorch.org/whl/cu126"
    ])

    P100_SETUP_MARKER.parent.mkdir(parents=True, exist_ok=True)
    P100_SETUP_MARKER.write_text("installed torch==2.11.0 cu126 for P100\n")
    verify_torch_install_in_subprocess()
    print("[INFO] Install finished. If torch import fails later, restart the kernel once.")
elif p100_detected and P100_SETUP_MARKER.exists():
    print("[INFO] P100 setup marker found. Continuing.")
else:
    print("[INFO] P100-specific PyTorch setup not needed for this GPU/session.")


In [ ]:
# =========================================================
# RUNTIME READY CHECK
# =========================================================

from pathlib import Path
import sys
import torch

IS_COLAB = "google.colab" in sys.modules or Path("/content").exists()
IS_KAGGLE = Path("/kaggle/input").exists()

print(f"[INFO] Colab environment: {IS_COLAB}")
print(f"[INFO] Kaggle environment: {IS_KAGGLE}")
print(f"[INFO] CUDA available: {torch.cuda.is_available()}")
print(f"[INFO] GPU count: {torch.cuda.device_count()}")
print(f"[INFO] PyTorch version: {torch.__version__}")
print(f"[INFO] PyTorch CUDA build: {torch.version.cuda}")
print(f"[INFO] PyTorch CUDA arch list: {torch.cuda.get_arch_list() if torch.cuda.is_available() else []}")

unsupported_gpus = []

for gpu_idx in range(torch.cuda.device_count()):
    gpu_name = torch.cuda.get_device_name(gpu_idx)
    capability = torch.cuda.get_device_capability(gpu_idx)
    arch_tag = f"sm_{capability[0]}{capability[1]}"
    supported_arches = torch.cuda.get_arch_list()

    print(f"[INFO] GPU {gpu_idx}: {gpu_name} | CUDA capability: {arch_tag}")

    if supported_arches and arch_tag not in supported_arches:
        unsupported_gpus.append(f"{gpu_name} ({arch_tag})")

if unsupported_gpus:
    print("\n[ERROR] This PyTorch build does not support the current GPU architecture.")
    print(f"[ERROR] Unsupported GPU(s): {unsupported_gpus}")

if IS_COLAB and not torch.cuda.is_available():
    print("\n[FIX] In Google Colab: Runtime > Change runtime type > Hardware accelerator > GPU, then rerun.")

if IS_KAGGLE:
    print("\n[INFO] Kaggle input datasets:")
    for item in Path("/kaggle/input").iterdir():
        print(f"- {item}")
elif IS_COLAB:
    print("\n[INFO] Running on Google Colab. Kaggle datasets will be downloaded with the Kaggle API if needed.")
else:
    print("\n[INFO] Running locally. Set DATASET_ROOT manually if datasets are not in the working directory.")


In [ ]:
# =========================================================
# COLAB KEEP-ALIVE HELPER
# =========================================================

# This reduces browser-side idle disconnects while the notebook is running.
# It cannot bypass Colab's hard runtime limits or quota limits.

KEEP_COLAB_AWAKE = True

if KEEP_COLAB_AWAKE:
    try:
        from IPython.display import Javascript, display

        display(Javascript("""
(function keepColabAwake() {
  const intervalMs = 60000;

  function clickConnectButton() {
    const candidates = Array.from(
      document.querySelectorAll('colab-connect-button, paper-button, mwc-button, button')
    );

    const connectButton = candidates.find((button) => {
      const text = (button.textContent || '').toLowerCase();
      return text.includes('connect') || text.includes('reconnect');
    });

    if (connectButton) {
      connectButton.click();
    }

    console.log('[Colab keep-alive] tick ' + new Date().toLocaleTimeString());
  }

  if (window.__codexColabKeepAliveInterval) {
    clearInterval(window.__codexColabKeepAliveInterval);
  }

  window.__codexColabKeepAliveInterval = setInterval(clickConnectButton, intervalMs);
  console.log('[Colab keep-alive] helper started. Hard runtime limits still apply.');
})();
"""))

        print("[INFO] Colab keep-alive helper started. Hard runtime/quota limits still apply.")
    except Exception as error:
        print(f"[WARN] Could not start Colab keep-alive helper: {error}")
else:
    print("[INFO] Colab keep-alive helper disabled.")


In [ ]:
# =========================================================
# DATASET NOTE FOR COLAB / KAGGLE
# =========================================================

# This notebook supports two dataset sources:
# 1. Kaggle dataset food41.
# 2. Your Tunisian food dataset from Google Drive.

# Sources used by default:
# - Kaggle: https://www.kaggle.com/datasets/kmader/food41
# - Drive : configure your own dataset locally.

# On Google Colab, the notebook will not ask you to upload files.
# It will try kagglehub for public Kaggle datasets.
# If credentials are needed, use Colab secrets KAGGLE_USERNAME and KAGGLE_KEY
# or set them as environment variables.

# Food101 is limited to 50 classes by default with FOOD101_MAX_CLASSES = 50.
# The notebook also limits images per class before training and builds:
# - Kaggle: /kaggle/working/combined_dataset
# - Colab : /content/working/combined_dataset

# Main values you can change in the configuration cell:
# - BATCH_SIZE = 32
# - EPOCHS = 35
# - FOOD101_MAX_CLASSES = 50
# - MAX_IMAGES_PER_CLASS = 500
# Changing BATCH_SIZE or EPOCHS does not rebuild or re-download datasets.
# Changing FOOD101_MAX_CLASSES or MAX_IMAGES_PER_CLASS rebuilds only the combined training subset;
# the raw Kaggle/Drive dataset cache is reused when available.


In [ ]:
# =========================================================
# IMPORTS + CONFIGURATION
# =========================================================

import os
import sys
import json
import itertools
import hashlib
import io
import shutil
import zipfile
from pathlib import Path

import numpy as np
import torch
import torchvision
import torch.nn as nn
import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None
    print("[WARN] tqdm is not installed. Progress bars will be disabled.")

from PIL import Image, ImageFile
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

ImageFile.LOAD_TRUNCATED_IMAGES = True

try:
    from sklearn.metrics import (
        auc,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
        roc_curve
    )
    from sklearn.preprocessing import label_binarize
except ImportError:
    print("[INFO] Installing scikit-learn...")
    os.system(f"{sys.executable} -m pip install -q scikit-learn")
    from sklearn.metrics import (
        auc,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
        roc_curve
    )
    from sklearn.preprocessing import label_binarize

try:
    from torchinfo import summary
except ImportError:
    summary = None
    print("[WARN] torchinfo is not installed. Model summary will be skipped.")

# =========================================================
# USER SETTINGS - EDIT THESE WITHOUT TOUCHING THE HELPERS
# =========================================================

SEED = 42
EPOCHS = 35
TRAIN_RATIO = 0.8
LEARNING_RATE = 1e-4
EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_MIN_DELTA = 1e-4
FINE_TUNE_LAST_BLOCKS = 2
MAX_IMAGES_PER_CLASS = 500
SELECTED_CLASSES = None
FOOD101_MAX_CLASSES = 50
FOOD101_SELECTED_CLASSES = None
DATASET_BUILD_VERSION = "food41_50_classes_500_images_plus_drive_tunisian_colab_gpu_v2"
RUN_MODEL_SUMMARY = False
SHOW_PROGRESS_BARS = True
COMPUTE_ROC_AUC = False
ROC_MAX_SAMPLES = 15000
MAX_CONFUSION_ANNOTATION_CLASSES = 35
REQUIRE_GPU = True

# Persistent Colab cache. This prevents re-downloading datasets after a runtime restart.
# It stores Kaggle downloads, sampled Drive images, combined dataset, models, and plots in Google Drive.
USE_PERSISTENT_COLAB_CACHE = True
AUTO_MOUNT_COLAB_DRIVE_FOR_CACHE = True
COLAB_CACHE_DIR = Path("/content/drive/MyDrive/food_training_cache")

# Google Colab Kaggle download options.
DOWNLOAD_KAGGLE_DATASETS_IN_COLAB = True
UPLOAD_KAGGLE_JSON_IN_COLAB = False
USE_KAGGLEHUB_FOR_PUBLIC_DATASETS = True
KAGGLE_DATASET_URLS = [
    "https://www.kaggle.com/datasets/kmader/food41"
]

# Kaggle API accepts dataset slugs such as "owner/dataset".
# The download helper also accepts full Kaggle dataset URLs.
KAGGLE_DATASETS = [
    "kmader/food41"
]

# Optional final override. If you already have one merged ImageFolder dataset,
# set DATASET_ROOT and the Kaggle/Colab merge logic will be skipped.
DATASET_ROOT = None
KAGGLE_DATASET_ROOT = None

# Google Drive is optional and disabled by default.
# For a large Drive dataset, paste a public ZIP file link into DRIVE_ZIP_URL.
DRIVE_FOLDER_URL = ""  # Configure your own dataset locally.
DRIVE_ZIP_URL = ""
DRIVE_DATASET_ROOT = None
ALLOW_SMALL_DRIVE_FOLDER_DOWNLOAD = False
USE_GOOGLE_DRIVE_API_FOR_FOLDER = True
DRIVE_API_MAX_SEARCH_DEPTH = 4
DRIVE_API_CLASS_RECURSION_DEPTH = 4

# Drive raw/source cache. Keep this >= MAX_IMAGES_PER_CLASS to avoid re-downloading
# the Drive dataset when you only reduce the training image limit later.
DRIVE_SOURCE_CACHE_IMAGES_PER_CLASS = 500

# Optional zip file override for Kaggle/local zip datasets.
ZIP_DATASET_PATH = None
AUTO_EXTRACT_ZIP = True
MAX_AUTO_ZIPS = 2

COMBINE_DATASETS = True
REBUILD_COMBINED_DATASET = False
LINK_DATASETS_INSTEAD_OF_COPY = False

IS_COLAB = "google.colab" in sys.modules or Path("/content").exists()
IS_KAGGLE = Path("/kaggle/input").exists()

def resolve_colab_working_dir():
    fallback_dir = Path("/content/working")

    if not USE_PERSISTENT_COLAB_CACHE:
        fallback_dir.mkdir(parents=True, exist_ok=True)
        return fallback_dir

    if AUTO_MOUNT_COLAB_DRIVE_FOR_CACHE:
        try:
            from google.colab import drive

            drive.mount("/content/drive", force_remount=False)
            COLAB_CACHE_DIR.mkdir(parents=True, exist_ok=True)
            print(f"[INFO] Using persistent Colab cache: {COLAB_CACHE_DIR}")
            return COLAB_CACHE_DIR
        except Exception as error:
            print(f"[WARN] Could not mount Google Drive cache: {error}")

    print(f"[WARN] Falling back to temporary Colab cache: {fallback_dir}")
    fallback_dir.mkdir(parents=True, exist_ok=True)
    return fallback_dir


if IS_KAGGLE:
    INPUT_DIR = Path("/kaggle/input")
    WORKING_DIR = Path("/kaggle/working")
elif IS_COLAB:
    INPUT_DIR = Path("/content")
    WORKING_DIR = resolve_colab_working_dir()
else:
    INPUT_DIR = Path(".")
    WORKING_DIR = Path(".")

MODEL_DIR = WORKING_DIR / "models"
PLOT_DIR = WORKING_DIR
COMBINED_DATASET_ROOT = WORKING_DIR / "combined_dataset"
DRIVE_DOWNLOAD_DIR = WORKING_DIR / "drive_dataset"
DRIVE_SAMPLED_DATASET_ROOT = DRIVE_DOWNLOAD_DIR / "sampled_drive_folder"
COLAB_KAGGLE_DATASET_DIR = WORKING_DIR / "kaggle_datasets"

MODEL_PATH = MODEL_DIR / "effnetb2_food101.pth"
HISTORY_PATH = MODEL_DIR / "history.json"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
COLAB_KAGGLE_DATASET_DIR.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET_ROOTS = [
    Path("/kaggle/input/datasets/kmader/food41"),
    Path("/kaggle/input/food41"),
    COLAB_KAGGLE_DATASET_DIR / "food41",
]

NUM_GPUS = torch.cuda.device_count()
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"[INFO] PyTorch version: {torch.__version__}")
print(f"[INFO] PyTorch CUDA build: {torch.version.cuda}")
print(f"[INFO] CUDA available: {torch.cuda.is_available()}")

unsupported_gpus = []

if device == "cuda":
    supported_arches = torch.cuda.get_arch_list()

    for gpu_idx in range(NUM_GPUS):
        gpu_name = torch.cuda.get_device_name(gpu_idx)
        capability = torch.cuda.get_device_capability(gpu_idx)
        arch_tag = f"sm_{capability[0]}{capability[1]}"

        print(f"[INFO] GPU {gpu_idx}: {gpu_name} | CUDA capability: {arch_tag}")

        if supported_arches and arch_tag not in supported_arches:
            unsupported_gpus.append(f"{gpu_name} ({arch_tag})")

if unsupported_gpus:
    raise RuntimeError(
        "This PyTorch build does not support the current GPU architecture: "
        f"{unsupported_gpus}. On Colab, switch to another GPU runtime or use Colab's default "
        "PyTorch environment. On Kaggle P100, enable the optional P100 setup cell."
    )

if REQUIRE_GPU and device != "cuda":
    raise RuntimeError(
        "GPU is required for this notebook. In Google Colab, open "
        "Runtime > Change runtime type > Hardware accelerator > GPU, then rerun all cells."
    )

BATCH_SIZE = 32
NUM_WORKERS = 2 if (IS_COLAB and device == "cuda") else 0
PERSISTENT_WORKERS = NUM_WORKERS > 0
PIN_MEMORY = device == "cuda"
USE_AMP = device == "cuda"
AMP_DEVICE_TYPE = "cuda" if device == "cuda" else "cpu"

if device == "cuda":
    torch.backends.cudnn.benchmark = True

print(f"[INFO] Environment: {'Colab' if IS_COLAB else 'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"[INFO] Using device: {device}")
print(f"[INFO] GPU count: {NUM_GPUS}")
print(f"[INFO] Batch size: {BATCH_SIZE}")
print(f"[INFO] Num workers: {NUM_WORKERS}")
print(f"[INFO] Persistent workers: {PERSISTENT_WORKERS}")
print(f"[INFO] Pin memory: {PIN_MEMORY}")
print(f"[INFO] Mixed precision AMP: {USE_AMP}")
print(f"[INFO] AMP device type: {AMP_DEVICE_TYPE}")
print(f"[INFO] Working directory: {WORKING_DIR}")
print(f"[INFO] Max images per class: {MAX_IMAGES_PER_CLASS}")
print(f"[INFO] Food101 max classes: {FOOD101_MAX_CLASSES}")
print(f"[INFO] Drive source cache images per class: {DRIVE_SOURCE_CACHE_IMAGES_PER_CLASS}")
print(f"[INFO] Persistent Colab cache: {USE_PERSISTENT_COLAB_CACHE}")


In [ ]:
# =========================================================
# DATASET HELPERS
# =========================================================

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def set_seeds(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)


def is_imagefolder_root(path, min_classes=1):
    path = Path(path)

    if not path.is_dir():
        return False

    class_dirs = [
        child
        for child in path.iterdir()
        if child.is_dir() and class_folder_has_images(child)
    ]

    return len(class_dirs) >= min_classes


def find_imagefolder_inside(root, min_classes=1):
    root = Path(root)

    if is_imagefolder_root(root, min_classes=min_classes):
        return root

    if not root.exists():
        return None

    for candidate in root.rglob("*"):
        if candidate.is_dir() and is_imagefolder_root(candidate, min_classes=min_classes):
            return candidate

    return None


def unique_paths(paths):
    seen = set()
    unique = []

    for path in paths:
        path = Path(path)
        key = str(path.resolve()) if path.exists() else str(path)

        if key not in seen:
            seen.add(key)
            unique.append(path)

    return unique


def candidate_kaggle_dataset_roots():
    candidates = []

    for configured_root in KAGGLE_DATASET_ROOTS:
        candidates.append(Path(configured_root))

    if KAGGLE_DATASET_ROOT is not None:
        candidates.append(Path(KAGGLE_DATASET_ROOT))

    if IS_KAGGLE:
        for dataset_dir in INPUT_DIR.iterdir():
            if not dataset_dir.is_dir():
                continue

            candidates.extend([
                dataset_dir,
                dataset_dir / "food101_60",
                dataset_dir / "food-101" / "images",
                dataset_dir / "images",
                dataset_dir / "train"
            ])

            for child in dataset_dir.iterdir():
                if child.is_dir():
                    candidates.append(child)
    elif IS_COLAB:
        for dataset_dir in COLAB_KAGGLE_DATASET_DIR.iterdir():
            if not dataset_dir.is_dir():
                continue

            candidates.extend([
                dataset_dir,
                dataset_dir / "food101_60",
                dataset_dir / "food-101" / "images",
                dataset_dir / "images",
                dataset_dir / "train"
            ])

            for child in dataset_dir.iterdir():
                if child.is_dir():
                    candidates.append(child)
    else:
        candidates.extend([
            Path("food101_60"),
            Path("new_dataset")
        ])

    return unique_paths(candidates)


def candidate_zip_files():
    if ZIP_DATASET_PATH is not None:
        return [Path(ZIP_DATASET_PATH)]

    search_roots = [Path(".")]

    if IS_KAGGLE:
        search_roots.append(INPUT_DIR)

    if IS_COLAB:
        search_roots.append(COLAB_KAGGLE_DATASET_DIR)

    zip_files = []

    for root in search_roots:
        if root.exists():
            zip_files.extend(sorted(root.rglob("*.zip")))

    return zip_files[:MAX_AUTO_ZIPS]


def find_dataset_root_in_extracted_zips():
    if not AUTO_EXTRACT_ZIP:
        return None

    for zip_path in candidate_zip_files():
        if not zip_path.exists():
            continue

        extract_dir = WORKING_DIR / zip_path.stem
        extract_dir.mkdir(parents=True, exist_ok=True)

        if not any(extract_dir.iterdir()):
            print(f"[INFO] Extracting zip dataset: {zip_path}")

            with zipfile.ZipFile(zip_path, "r") as zip_ref:
                zip_ref.extractall(extract_dir)

        extracted_root = find_imagefolder_inside(extract_dir, min_classes=1)

        if extracted_root is not None:
            return extracted_root

    return None


def normalize_kaggle_dataset_slug(dataset_reference):
    dataset_reference = str(dataset_reference).strip().rstrip("/")

    if "kaggle.com/datasets/" in dataset_reference:
        dataset_reference = dataset_reference.split("kaggle.com/datasets/", 1)[1]

    dataset_reference = dataset_reference.split("?", 1)[0].strip("/")
    parts = [part for part in dataset_reference.split("/") if part]

    if len(parts) < 2:
        raise ValueError(
            f"Invalid Kaggle dataset reference: {dataset_reference}. "
            "Expected a slug like 'owner/dataset' or a full Kaggle dataset URL."
        )

    return "/".join(parts[:2])


def import_kagglehub():
    try:
        import kagglehub
    except ImportError:
        print("[INFO] Installing kagglehub...")
        os.system(f"{sys.executable} -m pip install -q kagglehub")
        import kagglehub

    return kagglehub


def kaggle_credentials_available():
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    content_kaggle_json = Path("/content/kaggle.json")

    return (
        kaggle_json.exists()
        or content_kaggle_json.exists()
        or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
    )


def link_or_copy_directory(source_dir, destination_dir):
    source_dir = Path(source_dir)
    destination_dir = Path(destination_dir)

    if destination_dir.exists() and find_imagefolder_inside(destination_dir, min_classes=1) is not None:
        return

    if destination_dir.exists() and destination_dir.is_symlink():
        destination_dir.unlink()
    elif destination_dir.exists():
        shutil.rmtree(destination_dir)

    destination_dir.parent.mkdir(parents=True, exist_ok=True)

    # Copy raw public Kaggle data into the persistent cache so it survives Colab restarts.
    shutil.copytree(source_dir, destination_dir)


def download_public_kaggle_dataset_with_kagglehub(dataset_slug, dataset_dir):
    if not USE_KAGGLEHUB_FOR_PUBLIC_DATASETS:
        return False

    kagglehub = import_kagglehub()
    print(f"[INFO] Downloading public Kaggle dataset with kagglehub: {dataset_slug}")

    try:
        downloaded_path = Path(kagglehub.dataset_download(dataset_slug))
    except Exception as error:
        print(f"[WARN] kagglehub download failed for {dataset_slug}: {error}")
        return False

    if not downloaded_path.exists():
        print(f"[WARN] kagglehub returned a missing path: {downloaded_path}")
        return False

    link_or_copy_directory(downloaded_path, dataset_dir)
    print(f"[INFO] Kagglehub dataset available at: {dataset_dir}")
    return True


def import_kaggle_api():
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
    except ImportError:
        print("[INFO] Installing Kaggle API...")
        os.system(f"{sys.executable} -m pip install -q kaggle")
        from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()
    return api


def ensure_kaggle_credentials_for_colab():
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_json = kaggle_dir / "kaggle.json"
    content_kaggle_json = Path("/content/kaggle.json")

    if kaggle_json.exists():
        kaggle_json.chmod(0o600)
        return True

    if content_kaggle_json.exists():
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(content_kaggle_json, kaggle_json)
        kaggle_json.chmod(0o600)
        print(f"[INFO] Copied Kaggle credentials from {content_kaggle_json} to {kaggle_json}")
        return True

    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return True

    if IS_COLAB:
        try:
            from google.colab import userdata

            kaggle_username = userdata.get("KAGGLE_USERNAME")
            kaggle_key = userdata.get("KAGGLE_KEY")

            if kaggle_username and kaggle_key:
                os.environ["KAGGLE_USERNAME"] = kaggle_username
                os.environ["KAGGLE_KEY"] = kaggle_key
                print("[INFO] Loaded Kaggle credentials from Colab secrets.")
                return True
        except Exception:
            pass

    print(
        "[INFO] No Kaggle credentials found. The notebook will try kagglehub "
        "for public datasets without asking you to upload files."
    )
    return False


def download_colab_kaggle_datasets():
    if not IS_COLAB or not DOWNLOAD_KAGGLE_DATASETS_IN_COLAB:
        return

    COLAB_KAGGLE_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    has_kaggle_credentials = ensure_kaggle_credentials_for_colab()
    api = import_kaggle_api() if has_kaggle_credentials else None

    for dataset_reference in KAGGLE_DATASETS:
        dataset_slug = normalize_kaggle_dataset_slug(dataset_reference)
        dataset_name = dataset_slug.split("/")[-1]
        dataset_dir = COLAB_KAGGLE_DATASET_DIR / dataset_name

        if find_imagefolder_inside(dataset_dir, min_classes=1) is not None:
            print(f"[INFO] Reusing Colab Kaggle dataset: {dataset_dir}")
            continue

        if api is None:
            downloaded = download_public_kaggle_dataset_with_kagglehub(
                dataset_slug,
                dataset_dir
            )

            if downloaded:
                continue

            raise RuntimeError(
                "Could not download the Kaggle dataset without credentials. "
                "No file upload will be requested. Add KAGGLE_USERNAME and KAGGLE_KEY "
                "to Colab secrets, or set them as environment variables."
            )

        dataset_dir.mkdir(parents=True, exist_ok=True)
        print(f"[INFO] Downloading Kaggle dataset '{dataset_slug}' to {dataset_dir}")

        api.dataset_download_files(
            dataset_slug,
            path=str(dataset_dir),
            unzip=True,
            quiet=False
        )


def find_kaggle_dataset_roots():
    if IS_COLAB and DOWNLOAD_KAGGLE_DATASETS_IN_COLAB:
        download_colab_kaggle_datasets()

    detected_roots = []
    seen = set()

    for candidate in candidate_kaggle_dataset_roots():
        detected_root = find_imagefolder_inside(candidate, min_classes=1)

        if detected_root is None:
            continue

        key = str(detected_root.resolve())

        if key not in seen:
            seen.add(key)
            detected_roots.append(detected_root)

    if not detected_roots:
        zipped_root = find_dataset_root_in_extracted_zips()

        if zipped_root is not None:
            detected_roots.append(zipped_root)

    return detected_roots


def import_gdown():
    try:
        import gdown
        return gdown
    except ImportError:
        print("[INFO] Installing gdown...")
        os.system(f"{sys.executable} -m pip install -q gdown")
        import gdown
        return gdown


GOOGLE_DRIVE_FOLDER_MIME = "application/vnd.google-apps.folder"
DRIVE_CHILDREN_CACHE = {}


def extract_google_drive_folder_id(url):
    if not url:
        return None

    text = str(url).strip()

    if "/folders/" in text:
        return text.split("/folders/", 1)[1].split("?", 1)[0].split("/", 1)[0]

    if "id=" in text:
        return text.split("id=", 1)[1].split("&", 1)[0]

    return text


def sanitize_path_name(name):
    cleaned = "".join(
        char if char.isalnum() or char in "._- " else "_"
        for char in str(name)
    ).strip()

    return cleaned or "unnamed"


def drive_item_is_folder(item):
    return item.get("mimeType") == GOOGLE_DRIVE_FOLDER_MIME


def drive_item_is_image(item):
    name = item.get("name", "")
    mime_type = item.get("mimeType", "")

    return (
        mime_type.startswith("image/")
        or Path(name).suffix.lower() in IMAGE_EXTENSIONS
    )


def import_google_drive_service():
    if not IS_COLAB:
        raise RuntimeError(
            "Google Drive folder API mode is configured for Google Colab. "
            "For Kaggle/local, use DRIVE_ZIP_URL or set DRIVE_DATASET_ROOT manually."
        )

    try:
        from google.colab import auth
        auth.authenticate_user()
    except Exception as error:
        raise RuntimeError(
            "Google Colab authentication is required to read the Drive folder."
        ) from error

    try:
        from googleapiclient.discovery import build
    except ImportError:
        print("[INFO] Installing Google API client...")
        os.system(f"{sys.executable} -m pip install -q google-api-python-client")
        from googleapiclient.discovery import build

    return build("drive", "v3")


def list_drive_children(service, folder_id):
    folder_id = str(folder_id)

    if folder_id in DRIVE_CHILDREN_CACHE:
        return DRIVE_CHILDREN_CACHE[folder_id]

    children = []
    page_token = None

    while True:
        response = service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, mimeType, size)",
            pageSize=1000,
            pageToken=page_token,
            includeItemsFromAllDrives=True,
            supportsAllDrives=True
        ).execute()

        children.extend(response.get("files", []))
        page_token = response.get("nextPageToken")

        if not page_token:
            break

    DRIVE_CHILDREN_CACHE[folder_id] = children
    return children


def drive_folder_has_direct_images(service, folder_id):
    return any(
        drive_item_is_image(item)
        for item in list_drive_children(service, folder_id)
        if not drive_item_is_folder(item)
    )


def drive_folder_is_imagefolder_root(service, folder_id):
    for item in list_drive_children(service, folder_id):
        if drive_item_is_folder(item) and drive_folder_has_direct_images(service, item["id"]):
            return True

    return False


def find_drive_imagefolder_root_id(service, folder_id, max_depth=4):
    queue = [(folder_id, 0)]
    seen = set()

    while queue:
        current_id, depth = queue.pop(0)

        if current_id in seen:
            continue

        seen.add(current_id)

        if drive_folder_is_imagefolder_root(service, current_id):
            return current_id

        if depth >= max_depth:
            continue

        for item in list_drive_children(service, current_id):
            if drive_item_is_folder(item):
                queue.append((item["id"], depth + 1))

    return None


def list_drive_image_files_recursive(service, folder_id, max_depth=4):
    image_files = []
    queue = [(folder_id, 0)]
    seen = set()

    while queue:
        current_id, depth = queue.pop(0)

        if current_id in seen:
            continue

        seen.add(current_id)

        for item in list_drive_children(service, current_id):
            if drive_item_is_folder(item):
                if depth < max_depth:
                    queue.append((item["id"], depth + 1))
            elif drive_item_is_image(item):
                image_files.append(item)

    return image_files


def list_drive_class_folders(service, imagefolder_root_id):
    class_folders = []
    allowed_classes = selected_classes_set()

    for item in sorted(list_drive_children(service, imagefolder_root_id), key=lambda value: value.get("name", "")):
        if not drive_item_is_folder(item):
            continue

        class_name = item.get("name", "unnamed")

        if allowed_classes is not None and class_name not in allowed_classes:
            continue

        image_files = list_drive_image_files_recursive(
            service,
            item["id"],
            max_depth=DRIVE_API_CLASS_RECURSION_DEPTH
        )

        if image_files:
            class_folders.append((class_name, item["id"], image_files))

    return class_folders


def sample_drive_files_for_class(class_name, image_files):
    stable_hash = hashlib.md5(f"drive_{SEED}_{class_name}".encode("utf-8")).hexdigest()
    class_seed = int(stable_hash[:8], 16)
    rng = np.random.default_rng(class_seed)

    image_files = sorted(image_files, key=lambda item: item.get("name", ""))

    source_cache_limit = DRIVE_SOURCE_CACHE_IMAGES_PER_CLASS

    if source_cache_limit is not None and len(image_files) > source_cache_limit:
        selected_indices = rng.choice(
            len(image_files),
            size=source_cache_limit,
            replace=False
        )
        selected_indices.sort()
        selected_files = [image_files[index] for index in selected_indices]
    else:
        selected_files = image_files

    print(
        f"[INFO] Drive class '{class_name}': "
        f"using {len(selected_files)}/{len(image_files)} images"
    )

    return selected_files


def download_drive_file(service, file_id, destination_path):
    if destination_path.exists():
        return

    from googleapiclient.http import MediaIoBaseDownload

    request = service.files().get_media(fileId=file_id)
    destination_path.parent.mkdir(parents=True, exist_ok=True)

    with open(destination_path, "wb") as file_handle:
        downloader = MediaIoBaseDownload(file_handle, request)
        done = False

        while not done:
            _, done = downloader.next_chunk()


def drive_sample_metadata_path(path):
    return Path(path) / "_drive_sample_config.json"


def drive_sample_config(folder_id):
    return {
        "folder_id": folder_id,
        "source_cache_images_per_class": DRIVE_SOURCE_CACHE_IMAGES_PER_CLASS,
        "selected_classes": None if SELECTED_CLASSES is None else sorted(selected_classes_set()),
        "search_depth": DRIVE_API_MAX_SEARCH_DEPTH,
        "class_recursion_depth": DRIVE_API_CLASS_RECURSION_DEPTH
    }


def drive_sample_dataset_valid(path, folder_id):
    path = Path(path)
    metadata_path = drive_sample_metadata_path(path)

    if not is_imagefolder_root(path, min_classes=1) or not metadata_path.exists():
        return False

    try:
        with open(metadata_path, "r") as f:
            existing_config = json.load(f)
    except (json.JSONDecodeError, OSError):
        return False

    return existing_config == drive_sample_config(folder_id)


def download_drive_folder_sampled_dataset(url):
    folder_id = extract_google_drive_folder_id(url)

    if folder_id is None:
        return None

    if drive_sample_dataset_valid(DRIVE_SAMPLED_DATASET_ROOT, folder_id):
        print(f"[INFO] Reusing sampled Google Drive dataset: {DRIVE_SAMPLED_DATASET_ROOT}")
        return DRIVE_SAMPLED_DATASET_ROOT

    service = import_google_drive_service()
    imagefolder_root_id = find_drive_imagefolder_root_id(
        service,
        folder_id,
        max_depth=DRIVE_API_MAX_SEARCH_DEPTH
    )

    if imagefolder_root_id is None:
        raise FileNotFoundError(
            "Could not find an ImageFolder structure inside the Google Drive folder. "
            "Expected a folder containing class subfolders with images."
        )

    if DRIVE_SAMPLED_DATASET_ROOT.exists():
        safe_reset_directory(DRIVE_SAMPLED_DATASET_ROOT)
    else:
        DRIVE_SAMPLED_DATASET_ROOT.mkdir(parents=True, exist_ok=True)

    class_folders = list_drive_class_folders(service, imagefolder_root_id)

    if not class_folders:
        raise FileNotFoundError(
            "No class folders with images were found in the Google Drive folder."
        )

    for class_name, class_folder_id, image_files in class_folders:
        selected_files = sample_drive_files_for_class(class_name, image_files)
        class_dir = DRIVE_SAMPLED_DATASET_ROOT / sanitize_path_name(class_name)
        class_dir.mkdir(parents=True, exist_ok=True)

        for file_index, file_info in enumerate(selected_files):
            original_name = sanitize_path_name(file_info.get("name", f"image_{file_index}.jpg"))
            suffix = Path(original_name).suffix.lower()

            if suffix not in IMAGE_EXTENSIONS:
                suffix = ".jpg"

            destination_path = class_dir / f"drive_{file_index:08d}_{file_info['id']}{suffix}"
            download_drive_file(service, file_info["id"], destination_path)

    with open(drive_sample_metadata_path(DRIVE_SAMPLED_DATASET_ROOT), "w") as f:
        json.dump(drive_sample_config(folder_id), f, indent=4)

    sampled_dataset = ImageFolder(root=DRIVE_SAMPLED_DATASET_ROOT)
    print(f"[INFO] Sampled Drive dataset root: {DRIVE_SAMPLED_DATASET_ROOT}")
    print(f"[INFO] Sampled Drive classes: {len(sampled_dataset.classes)}")
    print(f"[INFO] Sampled Drive images: {len(sampled_dataset.samples)}")

    return DRIVE_SAMPLED_DATASET_ROOT


def download_drive_zip(gdown, url):
    DRIVE_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    drive_file = WORKING_DIR / "drive_dataset.zip"

    print(f"[INFO] Downloading Google Drive ZIP to: {drive_file}")
    downloaded = gdown.download(
        url=url,
        output=str(drive_file),
        quiet=False,
        fuzzy=True
    )

    if downloaded is None:
        raise RuntimeError(
            "Google Drive ZIP download failed. Make sure the link is public and internet is enabled."
        )

    downloaded_path = Path(downloaded)

    if not zipfile.is_zipfile(downloaded_path):
        raise RuntimeError(
            "The Drive file is not a valid ZIP archive. Zip the dataset folder and share the ZIP file link."
        )

    if not any(DRIVE_DOWNLOAD_DIR.iterdir()):
        print(f"[INFO] Extracting Google Drive ZIP to: {DRIVE_DOWNLOAD_DIR}")

        with zipfile.ZipFile(downloaded_path, "r") as zip_ref:
            zip_ref.extractall(DRIVE_DOWNLOAD_DIR)


def download_small_drive_folder(gdown, url):
    if not ALLOW_SMALL_DRIVE_FOLDER_DOWNLOAD:
        raise RuntimeError(
            "The provided Google Drive link is a folder. gdown cannot download large Drive folders "
            "with thousands of files. Zip the Drive folder first, share the ZIP file, then paste its "
            "link into DRIVE_ZIP_URL. Keep DRIVE_FOLDER_URL only as a reference."
        )

    DRIVE_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

    try:
        print(f"[INFO] Downloading small Google Drive folder to: {DRIVE_DOWNLOAD_DIR}")
        gdown.download_folder(
            url=url,
            output=str(DRIVE_DOWNLOAD_DIR),
            quiet=False
        )
    except Exception as error:
        message = str(error)
        class_name = error.__class__.__name__

        if "FolderContentsMaximumLimitError" in class_name or "more than 50 files" in message:
            raise RuntimeError(
                "Google Drive folder has too many files for gdown.download_folder. "
                "Zip the folder and put the ZIP file link in DRIVE_ZIP_URL."
            ) from error

        raise


def download_drive_dataset():
    if DRIVE_DATASET_ROOT is not None:
        drive_root = Path(DRIVE_DATASET_ROOT)
        detected_root = find_imagefolder_inside(drive_root, min_classes=1)

        if detected_root is not None:
            return detected_root

        raise FileNotFoundError(f"DRIVE_DATASET_ROOT is not an ImageFolder dataset: {drive_root}")

    if not DRIVE_ZIP_URL and not DRIVE_FOLDER_URL:
        return None

    if DRIVE_FOLDER_URL and USE_GOOGLE_DRIVE_API_FOR_FOLDER and IS_COLAB:
        return download_drive_folder_sampled_dataset(DRIVE_FOLDER_URL)

    gdown = import_gdown()

    if DRIVE_ZIP_URL:
        download_drive_zip(gdown, DRIVE_ZIP_URL)
    elif DRIVE_FOLDER_URL:
        download_small_drive_folder(gdown, DRIVE_FOLDER_URL)

    drive_root = find_imagefolder_inside(DRIVE_DOWNLOAD_DIR, min_classes=1)

    if drive_root is None:
        raise FileNotFoundError(
            "Downloaded Drive dataset was not found in ImageFolder format. "
            "Expected: root/class_name/image.jpg"
        )

    return drive_root


def safe_reset_directory(path):
    path = Path(path)
    working_root = WORKING_DIR.resolve()

    try:
        path.resolve().relative_to(working_root)
        is_safe = True
    except ValueError:
        is_safe = False

    if path.exists():
        if not is_safe:
            raise ValueError(f"Refusing to clear a directory outside WORKING_DIR: {path}")

        shutil.rmtree(path)

    path.mkdir(parents=True, exist_ok=True)


def link_or_copy_file(source_path, destination_path):
    source_path = Path(source_path)
    destination_path = Path(destination_path)

    if destination_path.exists():
        return

    if LINK_DATASETS_INSTEAD_OF_COPY:
        try:
            os.symlink(source_path, destination_path)
            return
        except OSError:
            pass

    shutil.copy2(source_path, destination_path)


def selected_classes_set():
    if SELECTED_CLASSES is None:
        return None

    return {str(class_name) for class_name in SELECTED_CLASSES}


def direct_image_files(folder):
    folder = Path(folder)

    for file_path in folder.iterdir():
        if file_path.is_file() and file_path.suffix.lower() in IMAGE_EXTENSIONS:
            yield file_path


def iter_image_files(folder):
    folder = Path(folder)

    for file_path in folder.rglob("*"):
        if file_path.is_file() and file_path.suffix.lower() in IMAGE_EXTENSIONS:
            yield file_path


def class_folder_has_images(folder):
    try:
        return next(direct_image_files(folder), None) is not None
    except PermissionError:
        return False


def dataset_build_config():
    return {
        "version": DATASET_BUILD_VERSION,
        "source_cache_images_per_class": DRIVE_SOURCE_CACHE_IMAGES_PER_CLASS,
        "selected_classes": None if SELECTED_CLASSES is None else sorted(selected_classes_set()),
        "food101_max_classes": FOOD101_MAX_CLASSES,
        "food101_selected_classes": (
            None
            if FOOD101_SELECTED_CLASSES is None
            else sorted(str(class_name) for class_name in FOOD101_SELECTED_CLASSES)
        ),
        "drive_folder_url": DRIVE_FOLDER_URL,
        "kaggle_datasets": [
            normalize_kaggle_dataset_slug(dataset_reference)
            for dataset_reference in KAGGLE_DATASETS
        ]
    }


def dataset_build_config_path(path):
    return Path(path) / "_build_config.json"


def combined_dataset_metadata_valid(path):
    metadata_path = dataset_build_config_path(path)

    if not metadata_path.exists():
        print("[INFO] Existing combined dataset has no build metadata; rebuilding.")
        return False

    try:
        with open(metadata_path, "r") as f:
            existing_config = json.load(f)
    except (json.JSONDecodeError, OSError):
        print("[INFO] Existing combined dataset metadata is invalid; rebuilding.")
        return False

    expected_config = dataset_build_config()

    if existing_config != expected_config:
        print("[INFO] Existing combined dataset config changed; rebuilding.")
        print(f"[INFO] Existing config: {existing_config}")
        print(f"[INFO] Expected config: {expected_config}")
        return False

    return True


def combined_dataset_within_image_limit(path):
    if not combined_dataset_metadata_valid(path):
        return False

    if MAX_IMAGES_PER_CLASS is None:
        return True

    if not is_imagefolder_root(path, min_classes=1):
        return False

    for class_dir in Path(path).iterdir():
        if not class_dir.is_dir():
            continue

        count = 0

        for _ in iter_image_files(class_dir):
            count += 1

            if count > MAX_IMAGES_PER_CLASS:
                print(
                    "[INFO] Existing combined dataset exceeds "
                    f"MAX_IMAGES_PER_CLASS={MAX_IMAGES_PER_CLASS}; rebuilding."
                )
                return False

    return True


def food101_selected_classes_set():
    if FOOD101_SELECTED_CLASSES is None:
        return None

    return {str(class_name) for class_name in FOOD101_SELECTED_CLASSES}


def source_is_food101(source_name):
    return str(source_name).startswith("kaggle")


def filter_food101_class_dirs(source_name, class_dirs):
    if not source_is_food101(source_name):
        return class_dirs

    selected_food101_classes = food101_selected_classes_set()

    if selected_food101_classes is not None:
        filtered_dirs = [
            class_dir
            for class_dir in class_dirs
            if class_dir.name in selected_food101_classes
        ]
        missing_classes = sorted(
            selected_food101_classes - {class_dir.name for class_dir in filtered_dirs}
        )

        if missing_classes:
            print(f"[WARN] FOOD101_SELECTED_CLASSES not found: {missing_classes}")
    elif FOOD101_MAX_CLASSES is not None:
        filtered_dirs = class_dirs[:FOOD101_MAX_CLASSES]
    else:
        filtered_dirs = class_dirs

    print(
        f"[INFO] Food101 source '{source_name}': "
        f"using {len(filtered_dirs)}/{len(class_dirs)} classes"
    )

    if filtered_dirs:
        print(
            "[INFO] Food101 classes selected: "
            + ", ".join(class_dir.name for class_dir in filtered_dirs)
        )

    return filtered_dirs


def list_class_dirs(source_roots):
    class_dirs_by_name = {}
    allowed_classes = selected_classes_set()

    for source_name, source_root in source_roots:
        source_root = Path(source_root)
        source_class_dirs = []

        for class_dir in sorted(source_root.iterdir()):
            if not class_dir.is_dir():
                continue

            class_name = class_dir.name

            if allowed_classes is not None and class_name not in allowed_classes:
                continue

            if not class_folder_has_images(class_dir):
                continue

            source_class_dirs.append(class_dir)

        source_class_dirs = filter_food101_class_dirs(source_name, source_class_dirs)

        for class_dir in source_class_dirs:
            class_name = class_dir.name
            class_dirs_by_name.setdefault(class_name, []).append((source_name, class_dir))

    if allowed_classes is not None:
        missing_classes = sorted(allowed_classes - set(class_dirs_by_name.keys()))

        if missing_classes:
            print(f"[WARN] Selected classes not found: {missing_classes}")

    return class_dirs_by_name


def reservoir_sample_class(class_name, source_class_dirs):
    max_images = MAX_IMAGES_PER_CLASS
    stable_hash = hashlib.md5(f"{SEED}_{class_name}".encode("utf-8")).hexdigest()
    class_seed = int(stable_hash[:8], 16)
    rng = np.random.default_rng(class_seed)
    selected_samples = []
    total_seen = 0

    for source_name, class_dir in source_class_dirs:
        for image_path in iter_image_files(class_dir):
            total_seen += 1
            sample = (source_name, total_seen, image_path)

            if max_images is None or len(selected_samples) < max_images:
                selected_samples.append(sample)
            else:
                replace_idx = rng.integers(0, total_seen)

                if replace_idx < max_images:
                    selected_samples[replace_idx] = sample

    selected_samples.sort(key=lambda item: str(item[2]))

    print(
        f"[INFO] Class '{class_name}': "
        f"using {len(selected_samples)}/{total_seen} images"
    )

    return selected_samples


def add_selected_samples_to_combined(class_name, selected_samples, combined_root):
    destination_dir = Path(combined_root) / class_name
    destination_dir.mkdir(parents=True, exist_ok=True)

    for sample_idx, (source_name, seen_idx, image_path) in enumerate(selected_samples):
        suffix = image_path.suffix.lower() or ".jpg"
        destination_name = f"{source_name}_{sample_idx:08d}_{seen_idx:08d}{suffix}"
        destination_path = destination_dir / destination_name

        link_or_copy_file(image_path, destination_path)


def build_combined_dataset(source_roots):
    if (
        COMBINED_DATASET_ROOT.exists()
        and not REBUILD_COMBINED_DATASET
        and combined_dataset_within_image_limit(COMBINED_DATASET_ROOT)
    ):
        combined_dataset = ImageFolder(root=COMBINED_DATASET_ROOT)
        print(f"[INFO] Reusing combined dataset: {COMBINED_DATASET_ROOT}")
        print(f"[INFO] Combined classes: {len(combined_dataset.classes)}")
        print(f"[INFO] Combined images: {len(combined_dataset.samples)}")
        return COMBINED_DATASET_ROOT

    safe_reset_directory(COMBINED_DATASET_ROOT)

    class_dirs_by_name = list_class_dirs(source_roots)

    for class_name in sorted(class_dirs_by_name.keys()):
        selected_samples = reservoir_sample_class(
            class_name=class_name,
            source_class_dirs=class_dirs_by_name[class_name]
        )
        add_selected_samples_to_combined(
            class_name=class_name,
            selected_samples=selected_samples,
            combined_root=COMBINED_DATASET_ROOT
        )

    with open(dataset_build_config_path(COMBINED_DATASET_ROOT), "w") as f:
        json.dump(dataset_build_config(), f, indent=4)

    combined_dataset = ImageFolder(root=COMBINED_DATASET_ROOT)

    print(f"[INFO] Combined dataset root: {COMBINED_DATASET_ROOT}")
    print(f"[INFO] Combined classes: {len(combined_dataset.classes)}")
    print(f"[INFO] Combined images: {len(combined_dataset.samples)}")

    return COMBINED_DATASET_ROOT

def prepare_dataset_root():
    if DATASET_ROOT is not None:
        dataset_root = Path(DATASET_ROOT)
        detected_root = find_imagefolder_inside(dataset_root, min_classes=1)

        if detected_root is None:
            raise FileNotFoundError(f"DATASET_ROOT is not an ImageFolder dataset: {dataset_root}")

        return detected_root

    source_roots = []

    kaggle_roots = find_kaggle_dataset_roots()

    if kaggle_roots:
        for dataset_idx, kaggle_root in enumerate(kaggle_roots, start=1):
            print(f"[INFO] Kaggle dataset root {dataset_idx}: {kaggle_root}")
            source_roots.append((f"kaggle_{dataset_idx}", kaggle_root))
    else:
        print("[WARN] No Kaggle ImageFolder dataset detected.")

    drive_root = download_drive_dataset()

    if drive_root is not None:
        print(f"[INFO] Google Drive dataset root: {drive_root}")
        source_roots.append(("drive", drive_root))
    else:
        print("[INFO] No Google Drive dataset URL configured.")

    if not source_roots:
        raise FileNotFoundError(
            "No dataset found. On Colab, provide Kaggle credentials so the datasets can be downloaded. "
            "On Kaggle, add datasets with Add Input. Alternatively set DRIVE_ZIP_URL or DATASET_ROOT manually."
        )

    if len(source_roots) == 1 and not COMBINE_DATASETS and MAX_IMAGES_PER_CLASS is None:
        return source_roots[0][1]

    return build_combined_dataset(source_roots)

In [ ]:
# =========================================================
# MODEL HELPERS
# =========================================================


def create_effnetb2_model(num_classes, seed=42, use_pretrained=True):
    torch.manual_seed(seed)

    fallback_transforms = torchvision.transforms.Compose([
        torchvision.transforms.Resize((224, 224)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    if use_pretrained:
        try:
            weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
            transforms = weights.transforms()
            model = torchvision.models.efficientnet_b2(weights=weights)
            print("[INFO] Loaded ImageNet pretrained EfficientNet_B2 weights.")
        except Exception as error:
            print(f"[WARN] Could not load pretrained weights: {error}")
            print("[WARN] Training EfficientNet_B2 from scratch. Enable Kaggle internet for better results.")
            transforms = fallback_transforms
            model = torchvision.models.efficientnet_b2(weights=None)
    else:
        transforms = fallback_transforms
        model = torchvision.models.efficientnet_b2(weights=None)

    for param in model.parameters():
        param.requires_grad = False

    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features=1408, out_features=num_classes)
    )

    return model, transforms


def strip_module_prefix(state_dict):
    return {
        key.replace("module.", "", 1): value
        for key, value in state_dict.items()
    }


def load_model_if_available(model, model_path):
    if not model_path.exists():
        print("[INFO] No saved model found. Training from scratch.")
        return model

    print(f"[INFO] Loading existing model: {model_path}")
    checkpoint = torch.load(model_path, map_location="cpu")

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        checkpoint = checkpoint["model_state_dict"]

    checkpoint = strip_module_prefix(checkpoint)
    model.load_state_dict(checkpoint)

    return model


def get_base_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def save_training_plot(values, labels, title, ylabel, output_path):
    plt.figure(figsize=(12, 5))

    for value, label in zip(values, labels):
        plt.plot(epochs, value, label=label)

    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"[INFO] Saved plot: {output_path}")

In [ ]:
# =========================================================
# PREPARE DATASET, MODEL, AND DATALOADERS
# =========================================================

required_prepare_objects = [
    "set_seeds",
    "prepare_dataset_root",
    "create_effnetb2_model",
    "load_model_if_available",
    "get_base_model",
    "ImageFolder",
    "DataLoader",
    "random_split",
    "device"
]
missing_prepare_objects = [name for name in required_prepare_objects if name not in globals()]

if missing_prepare_objects:
    raise RuntimeError(
        "Run the import/config, dataset helper, and model helper cells before this cell. "
        f"Missing objects: {missing_prepare_objects}"
    )

set_seeds(SEED)

food_root = prepare_dataset_root()
print(f"[INFO] Final dataset root: {food_root}")

dataset_without_transform = ImageFolder(root=food_root)
class_names = dataset_without_transform.classes

print(f"[INFO] Classes: {len(class_names)}")
print(f"[INFO] Total images: {len(dataset_without_transform)}")

base_model, effnetb2_transforms = create_effnetb2_model(
    num_classes=len(class_names),
    seed=SEED,
    use_pretrained=True
)

base_model = load_model_if_available(base_model, MODEL_PATH)

for param in base_model.features[-FINE_TUNE_LAST_BLOCKS:].parameters():
    param.requires_grad = True

print(f"[INFO] Fine-tuning last {FINE_TUNE_LAST_BLOCKS} EfficientNet feature blocks.")

if RUN_MODEL_SUMMARY and summary is not None:
    try:
        print("\n[INFO] Model Summary:")
        summary(
            base_model,
            input_size=(1, 3, 224, 224),
            device="cpu"
        )
    except Exception as error:
        print(f"[WARN] Model summary skipped: {error}")

dataset = ImageFolder(
    root=food_root,
    transform=effnetb2_transforms
)

train_size = int(TRAIN_RATIO * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS
)

print(f"[INFO] Train images: {len(train_dataset)}")
print(f"[INFO] Test images: {len(test_dataset)}")
print(f"[INFO] Train batches: {len(train_loader)}")
print(f"[INFO] Test batches: {len(test_loader)}")

effnetb2 = base_model.to(device)

if NUM_GPUS > 1:
    print(f"[INFO] Enabling DataParallel on {NUM_GPUS} GPUs.")
    effnetb2 = nn.DataParallel(effnetb2)

model_device = next(get_base_model(effnetb2).parameters()).device
print(f"[INFO] Model parameter device: {model_device}")

if device == "cuda" and model_device.type != "cuda":
    raise RuntimeError(
        "The model is still on CPU even though CUDA was selected. "
        "Rerun this cell after enabling the GPU."
    )

optimizer = torch.optim.Adam(
    effnetb2.parameters(),
    lr=LEARNING_RATE
)

loss_fn = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler(AMP_DEVICE_TYPE, enabled=USE_AMP)

if HISTORY_PATH.exists():
    with open(HISTORY_PATH, "r") as f:
        history = json.load(f)

    print("[INFO] Loaded previous history")
else:
    history = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": [],
        "f1_score": [],
        "precision": [],
        "recall": []
    }

    print("[INFO] Starting new history")

results = {
    "train_loss": [],
    "train_acc": [],
    "test_loss": [],
    "test_acc": [],
    "f1_score": [],
    "precision": [],
    "recall": []
}


In [ ]:
# =========================================================
# GPU SANITY CHECK
# =========================================================

required_gpu_check_objects = [
    "effnetb2",
    "train_loader",
    "device",
    "USE_AMP",
    "AMP_DEVICE_TYPE",
    "get_base_model"
]
missing_gpu_check_objects = [name for name in required_gpu_check_objects if name not in globals()]

if missing_gpu_check_objects:
    raise RuntimeError(
        "Run the previous prepare cell before this GPU check. "
        f"Missing objects: {missing_gpu_check_objects}"
    )


def gpu_memory_report(prefix):
    if device != "cuda":
        print(f"{prefix} GPU memory unavailable because device={device}")
        return

    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    print(
        f"{prefix} GPU memory allocated={allocated:.2f} GB | "
        f"reserved={reserved:.2f} GB"
    )


print(f"[INFO] torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"[INFO] selected device: {device}")

if torch.cuda.is_available():
    for gpu_idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(gpu_idx)
        total_memory = props.total_memory / (1024 ** 3)
        print(
            f"[INFO] GPU {gpu_idx}: {props.name} | "
            f"capability=sm_{props.major}{props.minor} | "
            f"total_memory={total_memory:.2f} GB"
        )

if device != "cuda":
    raise RuntimeError(
        "GPU is not active. In Google Colab, open Runtime > Change runtime type > "
        "Hardware accelerator > GPU, then rerun all cells from the top."
    )

model_device = next(get_base_model(effnetb2).parameters()).device
print(f"[INFO] model parameter device: {model_device}")

if model_device.type != "cuda":
    raise RuntimeError(
        "Model is not on GPU. Rerun the prepare cell after enabling CUDA."
    )

X_check, y_check = next(iter(train_loader))
print(f"[INFO] sample batch before move: X={X_check.device}, y={y_check.device}")

X_check = X_check.to(device, non_blocking=True)
y_check = y_check.to(device, non_blocking=True)
print(f"[INFO] sample batch after move: X={X_check.device}, y={y_check.device}")

gpu_memory_report("[INFO] before sanity forward")

effnetb2.eval()

with torch.inference_mode():
    with torch.autocast(
        device_type=AMP_DEVICE_TYPE,
        dtype=torch.float16,
        enabled=USE_AMP
    ):
        logits_check = effnetb2(X_check)

torch.cuda.synchronize()

print(f"[INFO] sanity logits device: {logits_check.device}")
print(f"[INFO] sanity logits shape: {tuple(logits_check.shape)}")
gpu_memory_report("[INFO] after sanity forward")
print("[INFO] GPU sanity check passed: model and data are using CUDA.")


In [ ]:
# =========================================================
# TRAINING
# =========================================================

required_training_objects = [
    "effnetb2",
    "train_loader",
    "test_loader",
    "optimizer",
    "loss_fn",
    "scaler",
    "device",
    "USE_AMP",
    "AMP_DEVICE_TYPE",
    "get_base_model"
]
missing_training_objects = [name for name in required_training_objects if name not in globals()]

if missing_training_objects:
    raise RuntimeError(
        "Run the previous prepare and GPU sanity-check cells before training. "
        f"Missing objects: {missing_training_objects}"
    )

if "gpu_memory_report" not in globals():
    def gpu_memory_report(prefix):
        if device != "cuda":
            print(f"{prefix} GPU memory unavailable because device={device}")
            return

        allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved = torch.cuda.memory_reserved() / (1024 ** 3)
        print(
            f"{prefix} GPU memory allocated={allocated:.2f} GB | "
            f"reserved={reserved:.2f} GB"
        )


def iter_with_progress(iterable, description):
    if SHOW_PROGRESS_BARS and globals().get("tqdm") is not None:
        return tqdm(
            iterable,
            total=len(iterable),
            desc=description,
            leave=True,
            dynamic_ncols=True
        )

    return iterable


model_device = next(get_base_model(effnetb2).parameters()).device
print(f"[INFO] Training device: {device}")
print(f"[INFO] Model parameter device before training: {model_device}")

if device == "cuda" and model_device.type != "cuda":
    raise RuntimeError(
        "Training stopped because the model is on CPU while CUDA is selected. "
        "Rerun the prepare cell."
    )

if device != "cuda":
    print("[WARN] Training is running on CPU. GPU is not active in this session.")

gpu_memory_report("[INFO] training start")
print("[INFO] Training model...")

best_test_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    print(f"\n========== Epoch {epoch + 1}/{EPOCHS} ==========")

    effnetb2.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for X, y in iter_with_progress(
        train_loader,
        f"Epoch {epoch + 1}/{EPOCHS} train"
    ):
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        if epoch == 0 and train_total == 0:
            current_model_device = next(get_base_model(effnetb2).parameters()).device
            print(
                f"[INFO] First train batch device: X={X.device}, "
                f"y={y.device}, model={current_model_device}"
            )
            gpu_memory_report("[INFO] first train batch")

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=AMP_DEVICE_TYPE,
            dtype=torch.float16,
            enabled=USE_AMP
        ):
            y_pred = effnetb2(X)
            loss = loss_fn(y_pred, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

        preds = torch.argmax(y_pred, dim=1)
        train_correct += (preds == y).sum().item()
        train_total += len(y)

    train_loss /= len(train_loader)
    train_acc = train_correct / train_total

    effnetb2.eval()

    test_loss = 0
    test_correct = 0
    test_total = 0

    all_preds = []
    all_labels = []

    with torch.inference_mode():
        for X, y in iter_with_progress(
            test_loader,
            f"Epoch {epoch + 1}/{EPOCHS} valid"
        ):
            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.autocast(
                device_type=AMP_DEVICE_TYPE,
                dtype=torch.float16,
                enabled=USE_AMP
            ):
                test_logits = effnetb2(X)
                loss = loss_fn(test_logits, y)

            test_loss += loss.item()

            preds = torch.argmax(test_logits, dim=1)
            test_correct += (preds == y).sum().item()
            test_total += len(y)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    test_loss /= len(test_loader)
    test_acc = test_correct / test_total

    f1 = f1_score(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    precision = precision_score(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    results["train_loss"].append(train_loss)
    results["train_acc"].append(train_acc)
    results["test_loss"].append(test_loss)
    results["test_acc"].append(test_acc)
    results["f1_score"].append(f1)
    results["precision"].append(precision)
    results["recall"].append(recall)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Acc  : {train_acc:.4f}")
    print(f"Test Loss  : {test_loss:.4f}")
    print(f"Test Acc   : {test_acc:.4f}")
    print(f"F1 Score   : {f1:.4f}")
    print(f"Precision  : {precision:.4f}")
    print(f"Recall     : {recall:.4f}")
    gpu_memory_report(f"[INFO] epoch {epoch + 1} end")

    if test_loss < best_test_loss - EARLY_STOPPING_MIN_DELTA:
        best_test_loss = test_loss
        best_epoch = epoch + 1
        epochs_without_improvement = 0

        torch.save(
            get_base_model(effnetb2).state_dict(),
            MODEL_PATH
        )

        print(f"[INFO] Best model updated at epoch {best_epoch}: test_loss={best_test_loss:.4f}")
    else:
        epochs_without_improvement += 1
        print(
            f"[INFO] No improvement for {epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE} epochs."
        )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"[INFO] Early stopping at epoch {epoch + 1}.")
            break

for key in history.keys():
    history[key].extend(results[key])

with open(HISTORY_PATH, "w") as f:
    json.dump(history, f, indent=4)

print(f"\n[INFO] History saved to {HISTORY_PATH}")

if MODEL_PATH.exists():
    print(f"[INFO] Best model saved to {MODEL_PATH} from epoch {best_epoch}")
else:
    torch.save(
        get_base_model(effnetb2).state_dict(),
        MODEL_PATH
    )
    print(f"[INFO] Model saved to {MODEL_PATH}")

epochs = range(1, len(history["train_loss"]) + 1)

save_training_plot(
    values=[history["train_loss"], history["test_loss"]],
    labels=["Train Loss", "Test Loss"],
    title="Loss Curve",
    ylabel="Loss",
    output_path=PLOT_DIR / "loss_curve.png"
)

save_training_plot(
    values=[history["train_acc"], history["test_acc"]],
    labels=["Train Accuracy", "Test Accuracy"],
    title="Accuracy Curve",
    ylabel="Accuracy",
    output_path=PLOT_DIR / "accuracy_curve.png"
)

save_training_plot(
    values=[history["f1_score"]],
    labels=["F1 Score"],
    title="F1 Score Curve",
    ylabel="F1 Score",
    output_path=PLOT_DIR / "f1_score_curve.png"
)

print("\n[INFO] Training complete!")


In [ ]:
# =========================================================
# CONFUSION MATRIX HELPER
# =========================================================

def make_confusion_matrix(
    y_true,
    y_pred,
    classes=None,
    figsize=(20, 20),
    text_size=8,
    norm=False,
    savefig=False
):

    if classes:
        labels = classes
        cm = confusion_matrix(
            y_true,
            y_pred,
            labels=np.arange(len(classes))
        )
    else:
        labels = np.arange(len(np.unique(y_true)))
        cm = confusion_matrix(y_true, y_pred)

    with np.errstate(divide="ignore", invalid="ignore"):
        cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
        cm_norm = np.nan_to_num(cm_norm)

    n_classes = cm.shape[0]

    fig, ax = plt.subplots(figsize=figsize)
    cax = ax.matshow(cm, cmap=plt.cm.Blues)
    fig.colorbar(cax)

    ax.set(
        title="Confusion Matrix",
        xlabel="Predicted Label",
        ylabel="True Label",
        xticks=np.arange(n_classes),
        yticks=np.arange(n_classes),
        xticklabels=labels,
        yticklabels=labels
    )

    ax.xaxis.set_label_position("bottom")
    ax.xaxis.tick_bottom()
    plt.xticks(rotation=90)

    threshold = (cm.max() + cm.min()) / 2.
    annotate_cells = n_classes <= MAX_CONFUSION_ANNOTATION_CLASSES

    if not annotate_cells:
        print(
            f"[INFO] Skipping per-cell annotations for {n_classes} classes "
            f"to save memory/time."
        )

    if annotate_cells:
        for i, j in itertools.product(
            range(cm.shape[0]),
            range(cm.shape[1])
        ):
            if norm:
                text = f"{cm[i, j]}\n({cm_norm[i, j] * 100:.1f}%)"
            else:
                text = f"{cm[i, j]}"

            plt.text(
                j,
                i,
                text,
                horizontalalignment="center",
                color="white" if cm[i, j] > threshold else "black",
                size=text_size
            )

    plt.tight_layout()

    if savefig:
        output_dir = globals().get("PLOT_DIR", Path("."))
        output_path = output_dir / "confusion_matrix.png"

        plt.savefig(
            output_path,
            dpi=300,
            bbox_inches="tight"
        )

        print(f"[INFO] Confusion matrix saved to {output_path}")

    plt.show()


In [ ]:
# =========================================================
# CONFUSION MATRIX + ROC/AUC
# =========================================================

required_eval_objects = [
    "effnetb2",
    "test_loader",
    "class_names",
    "device",
    "USE_AMP",
    "AMP_DEVICE_TYPE",
    "get_base_model",
    "make_confusion_matrix"
]
missing_eval_objects = [name for name in required_eval_objects if name not in globals()]

if missing_eval_objects:
    raise RuntimeError(
        "Run the previous prepare/training cells before evaluation. "
        f"Missing objects: {missing_eval_objects}"
    )

model_device = next(get_base_model(effnetb2).parameters()).device
print(f"[INFO] Evaluation device: {device}")
print(f"[INFO] Model parameter device before evaluation: {model_device}")

PLOT_DIR = globals().get("PLOT_DIR", Path("."))
MODEL_DIR = globals().get("MODEL_DIR", Path("models"))
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def sample_roc_inputs(y_true, y_score, max_samples, seed=42):
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    if max_samples is None or len(y_true) <= max_samples:
        return y_true, y_score

    rng = np.random.default_rng(seed)
    selected_indices = rng.choice(len(y_true), size=max_samples, replace=False)
    selected_indices.sort()

    print(
        f"[INFO] Sampling {max_samples}/{len(y_true)} test predictions "
        "for ROC/AUC to save memory."
    )

    return y_true[selected_indices], y_score[selected_indices]


def plot_multiclass_roc_auc(
    y_true,
    y_score,
    classes,
    savefig=True
):

    n_classes = len(classes)
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    if len(np.unique(y_true)) < 2:
        print("[WARN] ROC/AUC needs at least two classes in y_true.")
        return {}

    plt.figure(figsize=(12, 10))

    if n_classes == 2:
        positive_class = 1
        y_binary = (y_true == positive_class).astype(int)

        fpr, tpr, _ = roc_curve(
            y_binary,
            y_score[:, positive_class]
        )

        roc_auc = auc(fpr, tpr)

        plt.plot(
            fpr,
            tpr,
            color="darkorange",
            lw=2,
            label=f"ROC curve (AUC = {roc_auc:.3f})"
        )

        auc_scores = {
            classes[positive_class]: float(roc_auc)
        }

        print("\n[INFO] ROC/AUC results:")
        print(f"AUC ({classes[positive_class]}): {roc_auc:.4f}")

    else:
        y_true_bin = label_binarize(
            y_true,
            classes=np.arange(n_classes)
        )

        fpr = {}
        tpr = {}
        roc_auc = {}
        valid_class_indices = []

        for class_idx in range(n_classes):
            if len(np.unique(y_true_bin[:, class_idx])) < 2:
                continue

            fpr[class_idx], tpr[class_idx], _ = roc_curve(
                y_true_bin[:, class_idx],
                y_score[:, class_idx]
            )

            roc_auc[class_idx] = auc(
                fpr[class_idx],
                tpr[class_idx]
            )

            valid_class_indices.append(class_idx)

        if not valid_class_indices:
            print("[WARN] ROC/AUC could not be computed.")
            plt.close()
            return {}

        fpr["micro"], tpr["micro"], _ = roc_curve(
            y_true_bin[:, valid_class_indices].ravel(),
            y_score[:, valid_class_indices].ravel()
        )

        roc_auc["micro"] = auc(
            fpr["micro"],
            tpr["micro"]
        )

        all_fpr = np.unique(
            np.concatenate(
                [
                    fpr[class_idx]
                    for class_idx in valid_class_indices
                ]
            )
        )

        mean_tpr = np.zeros_like(all_fpr)

        for class_idx in valid_class_indices:
            mean_tpr += np.interp(
                all_fpr,
                fpr[class_idx],
                tpr[class_idx]
            )

        mean_tpr /= len(valid_class_indices)

        fpr["macro"] = all_fpr
        tpr["macro"] = mean_tpr

        roc_auc["macro"] = auc(
            fpr["macro"],
            tpr["macro"]
        )

        show_class_legend = len(valid_class_indices) <= 15

        colors = plt.cm.tab20(
            np.linspace(
                0,
                1,
                max(len(valid_class_indices), 1)
            )
        )

        for class_idx, color in zip(valid_class_indices, colors):
            label = (
                f"{classes[class_idx]} (AUC = {roc_auc[class_idx]:.2f})"
                if show_class_legend
                else None
            )

            plt.plot(
                fpr[class_idx],
                tpr[class_idx],
                color=color,
                lw=1,
                alpha=0.35,
                label=label
            )

        if not show_class_legend:
            plt.plot(
                [],
                [],
                color="gray",
                lw=1,
                alpha=0.35,
                label="One-vs-rest class ROC curves"
            )

        plt.plot(
            fpr["micro"],
            tpr["micro"],
            color="deeppink",
            linestyle=":",
            linewidth=4,
            label=f"Micro-average ROC (AUC = {roc_auc['micro']:.3f})"
        )

        plt.plot(
            fpr["macro"],
            tpr["macro"],
            color="navy",
            linestyle=":",
            linewidth=4,
            label=f"Macro-average ROC (AUC = {roc_auc['macro']:.3f})"
        )

        auc_scores = {
            "micro_average": float(roc_auc["micro"]),
            "macro_average": float(roc_auc["macro"])
        }

        auc_scores.update(
            {
                classes[class_idx]: float(roc_auc[class_idx])
                for class_idx in valid_class_indices
            }
        )

        print("\n[INFO] ROC/AUC results:")
        print(f"Micro-average AUC: {roc_auc['micro']:.4f}")
        print(f"Macro-average AUC: {roc_auc['macro']:.4f}")

        for class_idx in valid_class_indices:
            print(f"{classes[class_idx]} AUC: {roc_auc[class_idx]:.4f}")

    plt.plot(
        [0, 1],
        [0, 1],
        color="black",
        linestyle="--",
        lw=1
    )

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves and AUC")
    plt.legend(loc="lower right", fontsize=8)
    plt.grid(alpha=0.3)
    plt.tight_layout()

    if savefig:
        roc_curve_path = PLOT_DIR / "roc_auc_curve.png"

        plt.savefig(
            roc_curve_path,
            dpi=300,
            bbox_inches="tight"
        )

    plt.show()

    roc_scores_path = MODEL_DIR / "roc_auc_scores.json"

    with open(roc_scores_path, "w") as f:
        json.dump(auc_scores, f, indent=4)

    if savefig:
        print(f"[INFO] ROC curve saved to {roc_curve_path}")

    print(f"[INFO] AUC scores saved to {roc_scores_path}")

    return auc_scores


print("[INFO] Creating confusion matrix and evaluation curves...")

effnetb2.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.inference_mode():
    for X, y in test_loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.autocast(
            device_type=AMP_DEVICE_TYPE,
            dtype=torch.float16,
            enabled=USE_AMP
        ):
            logits = effnetb2(X)

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

make_confusion_matrix(
    y_true=all_labels,
    y_pred=all_preds,
    classes=class_names,
    figsize=(30, 30),
    text_size=6,
    norm=True,
    savefig=True
)

if COMPUTE_ROC_AUC:
    roc_labels, roc_probs = sample_roc_inputs(
        y_true=all_labels,
        y_score=all_probs,
        max_samples=ROC_MAX_SAMPLES,
        seed=SEED
    )

    roc_auc_scores = plot_multiclass_roc_auc(
        y_true=roc_labels,
        y_score=roc_probs,
        classes=class_names,
        savefig=True
    )
else:
    roc_auc_scores = {}
    print("[INFO] ROC/AUC skipped because COMPUTE_ROC_AUC=False.")
